# Small-model routing lab: existing code before new code

This Kaggle notebook evaluates a constrained `GoalIR -> RecipeIR` routing harness over package operations at **subpackage and symbol granularity**. It prefers deterministic retrieval, compatibility, port checks, and route validation; an optional Hugging Face instruct model may propose structured goals or recipes, but its operation IDs are never trusted without deterministic validation.

The notebook first looks for notebook 02's polymorphic feature ledger and consumes its operation records, port features, blocking keys, field-specific vectors, and typed candidate edges. It can also use this repository's verified search projection or a compatible SQLite catalog. A small, explicitly marked fallback catalog keeps every cell runnable and supplies frozen routing tasks; it is an evaluation fixture, not evidence about ecosystem-scale quality.

Measured outputs are written under `/kaggle/working/reuse_routing_lab`. Dense and generative models are opt-in to keep the default run light; skipped or failed arms are recorded, and no result values are fabricated.

In [ ]:
# Kaggle images already include several of these; pip will reuse satisfied installs.
!pip install -q "torch" "packaging>=24" "pandas>=2.0" "pyarrow>=15" "scikit-learn>=1.4" "sentence-transformers>=3.0" "transformers>=5.10.1" "accelerate>=0.34"

## Configuration

Set environment variables in a Kaggle secret or edit this cell. The configurable GPU default is `google/gemma-4-E2B-it`, following Google's [Gemma 4 overview](https://ai.google.dev/gemma/docs/core) and [official Transformers thinking/inference guide](https://ai.google.dev/gemma/docs/capabilities/thinking). The official guide uses `transformers>=5.10.1`, `AutoProcessor`, and `AutoModelForMultimodalLM`. Access may require accepting the model license and supplying `HF_TOKEN`; the published memory estimates also make GPU/precision planning important. Set `USE_HF_ROUTER=1` only when you want the optional generation arm. A default run measures deterministic routing only and must not be described as tested model inference.

In [ ]:
from __future__ import annotations

from collections import defaultdict, deque
from dataclasses import asdict, dataclass, field
from importlib import metadata as importlib_metadata
from itertools import product
from pathlib import Path
from typing import Any, Iterable, Literal, Mapping, Sequence
import hashlib
import json
import os
import platform as platform_module
import re
import sqlite3
import subprocess
import sys
import time

import numpy as np
import pandas as pd
from packaging.specifiers import InvalidSpecifier, SpecifierSet
from packaging.version import Version
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def env_flag(name: str, default: bool = False) -> bool:
    raw = os.getenv(name)
    return default if raw is None else raw.strip().casefold() in {"1", "true", "yes", "on"}

OUTPUT_DIR = Path("/kaggle/working/reuse_routing_lab")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_LEDGER_HINT = os.getenv("REUSE_FEATURE_LEDGER_PATH", os.getenv("FEATURE_LEDGER_PATH", "")).strip()
BOOTSTRAP_REPO = env_flag("BOOTSTRAP_REPO", False)
GITHUB_REPO_URL = os.getenv("GITHUB_REPO_URL", "").strip()
REPO_REF = os.getenv("REPO_REF", "main").strip() or "main"
USE_HF_ROUTER = env_flag("USE_HF_ROUTER", False)
HF_MODEL_ID = os.getenv("HF_MODEL_ID", "google/gemma-4-E2B-it")
HF_MAX_NEW_TOKENS = int(os.getenv("HF_MAX_NEW_TOKENS", "384"))
RUN_DENSE_ABLATIONS = env_flag("RUN_DENSE_ABLATIONS", False)
DENSE_MODEL_IDS = tuple(filter(None, os.getenv("DENSE_MODEL_IDS", "sentence-transformers/all-MiniLM-L6-v2").split(",")))
DENSE_BATCH_SIZE = int(os.getenv("DENSE_BATCH_SIZE", "32"))
DENSE_DEVICE = os.getenv("DENSE_DEVICE", "cuda" if env_flag("USE_GPU", False) else "cpu")
ABLATION_SCALE = os.getenv("ABLATION_SCALE", "light").casefold()  # light | medium | full
TOP_K = int(os.getenv("TOP_K", "16"))
ABSTAIN_THRESHOLD = float(os.getenv("ABSTAIN_THRESHOLD", "0.12"))
PYTHON_VERSION = f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"
PLATFORM = {"darwin": "macos", "win32": "windows"}.get(sys.platform, "linux")

LAB_CONFIG = {
    "output_dir": str(OUTPUT_DIR),
    "feature_ledger_hint": FEATURE_LEDGER_HINT,
    "bootstrap_repo": BOOTSTRAP_REPO,
    "repo_ref": REPO_REF,
    "use_hf_router": USE_HF_ROUTER,
    "hf_model_id": HF_MODEL_ID,
    "run_dense_ablations": RUN_DENSE_ABLATIONS,
    "dense_model_ids": DENSE_MODEL_IDS,
    "dense_device": DENSE_DEVICE,
    "ablation_scale": ABLATION_SCALE,
    "top_k": TOP_K,
    "abstain_threshold": ABSTAIN_THRESHOLD,
    "python_version": PYTHON_VERSION,
    "platform": PLATFORM,
}
print(json.dumps(LAB_CONFIG, indent=2))

## Optional repository bootstrap and discovery

For a Kaggle notebook detached from the repository, set `BOOTSTRAP_REPO=1` and `GITHUB_REPO_URL=https://github.com/<owner>/<repo>.git`. The command uses an argument list rather than interpolating credentials into a shell. A checked-out repository, Kaggle dataset mount, or explicit `REUSE_REPO_DIR` takes precedence.

In [ ]:
CLONE_DIR = Path("/kaggle/working/this-already-exists-dont-rebuild-it")
if BOOTSTRAP_REPO:
    if not GITHUB_REPO_URL:
        raise ValueError("BOOTSTRAP_REPO=1 requires GITHUB_REPO_URL")
    if not CLONE_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_REF, GITHUB_REPO_URL, str(CLONE_DIR)],
            check=True,
        )

def discover_project_root() -> Path | None:
    candidates: list[Path] = []
    if os.getenv("REUSE_REPO_DIR"):
        candidates.append(Path(os.environ["REUSE_REPO_DIR"]))
    candidates.extend([CLONE_DIR, Path.cwd(), Path.cwd().parent])
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.extend(path for path in kaggle_input.iterdir() if path.is_dir())
    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        for root in (candidate, candidate / "project"):
            if (root / "pyproject.toml").exists() and (root / "src").exists():
                return root
    return None

PROJECT_ROOT = discover_project_root()
if PROJECT_ROOT is not None:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("project_root:", PROJECT_ROOT)

## Strict intermediate representations

The model boundary accepts only these records. A `RecipeIR` that names an unknown operation, violates ports, skips required stages, or conflicts with environment policy is rejected before execution. `no_reuse` means the catalog has no valid route under the stated constraints; `abstain` means the router lacks enough evidence to decide.

In [ ]:
Decision = Literal["reuse", "no_reuse", "abstain"]

def stable_digest(value: Any) -> str:
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), default=str).encode()
    return "sha256:" + hashlib.sha256(payload).hexdigest()

@dataclass(frozen=True)
class Operation:
    operation_id: str
    package_name: str
    package_version: str
    subpackage: str
    module: str
    symbol: str
    kind: str
    signature: str
    description: str
    input_types: tuple[str, ...]
    output_types: tuple[str, ...]
    stage: str
    labels: tuple[str, ...] = ()
    blocks: tuple[tuple[str, str], ...] = ()
    python_spec: str = ">=3.9"
    platforms: tuple[str, ...] = ("any",)
    effects: tuple[str, ...] = ()
    graph_next: tuple[str, ...] = ()
    evidence_level: str = "static_source"
    source: str = "fallback_fixture"

    def __post_init__(self) -> None:
        if not self.operation_id or not self.module or not self.symbol:
            raise ValueError("operation identity fields cannot be empty")
        if not self.output_types:
            raise ValueError(f"{self.operation_id}: at least one output port is required")

@dataclass(frozen=True)
class GoalIR:
    prompt: str
    initial_artifact: str
    required_output: str
    required_stages: tuple[str, ...]
    query_labels: tuple[str, ...] = ()
    query_blocks: tuple[tuple[str, str], ...] = ()
    python_version: str = PYTHON_VERSION
    platform: str = PLATFORM
    network_allowed: bool = False
    max_steps: int = 8

    def __post_init__(self) -> None:
        if not self.prompt.strip() or not self.initial_artifact or not self.required_output:
            raise ValueError("GoalIR prompt and artifact types are required")
        if not 1 <= self.max_steps <= 24:
            raise ValueError("GoalIR.max_steps must be in [1, 24]")
        Version(self.python_version)

    @classmethod
    def from_mapping(cls, value: Mapping[str, Any]) -> "GoalIR":
        allowed = {field.name for field in __import__("dataclasses").fields(cls)}
        unknown = set(value) - allowed
        if unknown:
            raise ValueError(f"unknown GoalIR fields: {sorted(unknown)}")
        data = dict(value)
        for key in ("required_stages", "query_labels"):
            if key in data:
                data[key] = tuple(str(item) for item in data[key])
        if "query_blocks" in data:
            data["query_blocks"] = tuple((str(a), str(b)) for a, b in data["query_blocks"])
        return cls(**data)

@dataclass(frozen=True)
class RecipeIR:
    decision: Decision
    operation_ids: tuple[str, ...] = ()
    confidence: float = 0.0
    proposer: str = "deterministic"
    rationale: str = ""

    def __post_init__(self) -> None:
        if self.decision not in {"reuse", "no_reuse", "abstain"}:
            raise ValueError(f"unsupported recipe decision: {self.decision}")
        if not 0.0 <= self.confidence <= 1.0:
            raise ValueError("RecipeIR.confidence must lie in [0, 1]")
        if self.decision == "reuse" and not self.operation_ids:
            raise ValueError("reuse requires at least one operation ID")
        if self.decision != "reuse" and self.operation_ids:
            raise ValueError("no_reuse/abstain recipes cannot contain operation IDs")

    @classmethod
    def from_mapping(cls, value: Mapping[str, Any], proposer: str) -> "RecipeIR":
        allowed = {"decision", "operation_ids", "confidence", "rationale"}
        unknown = set(value) - allowed
        if unknown:
            raise ValueError(f"unknown RecipeIR fields: {sorted(unknown)}")
        return cls(
            decision=str(value["decision"]),
            operation_ids=tuple(str(item) for item in value.get("operation_ids", ())),
            confidence=float(value.get("confidence", 0.0)),
            proposer=proposer,
            rationale=str(value.get("rationale", "")),
        )

@dataclass(frozen=True)
class Candidate:
    operation_id: str
    score: float
    rank: int
    subpackage: str
    provenance: Mapping[str, Any]

@dataclass(frozen=True)
class ValidationResult:
    valid: bool
    status: str
    reasons: tuple[str, ...]
    final_artifact: str | None = None
    covered_stages: tuple[str, ...] = ()

## Catalog loading with a self-contained fallback

Loader priority is: notebook 02 feature-ledger run, explicit/discovered SQLite catalog, repository verified-capability projection, then fallback fixture. Ledger rows remain keyed by exact operation ID and are projected into symbol labels, blocking keys, signature ports, typed graph edges, and optional upstream embedding planes. Missing port facts remain `unknown/*`; the notebook does not invent compatibility from docstrings. The fallback fixture is merged only when needed to retain the notebook's frozen executable routing tasks.

In [ ]:
def installed_version(distribution: str) -> str:
    try:
        return importlib_metadata.version(distribution)
    except importlib_metadata.PackageNotFoundError:
        return "unknown"

def fallback_catalog() -> list[Operation]:
    pandas_version = installed_version("pandas")
    stdlib_version = platform_module.python_version()
    def op(operation_id: str, package: str, version: str, module: str, symbol: str, signature: str,
           description: str, inputs: Sequence[str], outputs: Sequence[str], stage: str,
           labels: Sequence[str], next_ids: Sequence[str] = ()) -> Operation:
        subpackage = module.rsplit(".", 1)[0] if "." in module else module
        blocks = [("workflow_stage", stage)]
        blocks.extend(("input_artifact", item) for item in inputs)
        blocks.extend(("output_artifact", item) for item in outputs)
        return Operation(
            operation_id, package, version, subpackage, module, symbol, "function", signature,
            description, tuple(inputs), tuple(outputs), stage, tuple(labels), tuple(blocks),
            graph_next=tuple(next_ids), evidence_level="contract_tested_fixture",
        )
    return [
        op("pandas.read_csv", "pandas", pandas_version, "pandas.io.parsers", "read_csv", "(filepath_or_buffer, **kwargs) -> DataFrame", "Read comma-separated data into a pandas DataFrame.", ["file/csv"], ["table/pandas-dataframe"], "load", ["load csv", "csv dataframe"], ["pandas.DataFrame.fillna", "pandas.DataFrame.groupby", "pandas.DataFrame.to_json"]),
        op("pandas.read_json", "pandas", pandas_version, "pandas.io.json", "read_json", "(path_or_buf, **kwargs) -> DataFrame", "Read JSON data into a pandas DataFrame.", ["file/json"], ["table/pandas-dataframe"], "load", ["load json", "json dataframe"]),
        op("pandas.DataFrame.fillna", "pandas", pandas_version, "pandas.core.frame", "DataFrame.fillna", "(value=None, **kwargs) -> DataFrame", "Replace missing values in a DataFrame.", ["table/pandas-dataframe"], ["table/pandas-dataframe"], "missing-values", ["fill nulls", "missing values"], ["pandas.DataFrame.groupby", "pandas.DataFrame.to_json"]),
        op("pandas.DataFrame.groupby", "pandas", pandas_version, "pandas.core.frame", "DataFrame.groupby", "(by, **kwargs) -> DataFrameGroupBy", "Group DataFrame rows by one or more keys.", ["table/pandas-dataframe"], ["group/pandas-dataframe-groupby"], "group", ["group rows", "group by column"], ["pandas.core.groupby.DataFrameGroupBy.sum"]),
        op("pandas.core.groupby.DataFrameGroupBy.sum", "pandas", pandas_version, "pandas.core.groupby.generic", "DataFrameGroupBy.sum", "(**kwargs) -> Series", "Sum a numeric value within each DataFrame group.", ["group/pandas-dataframe-groupby"], ["series/pandas"], "aggregate", ["grouped sum", "sum by category"], ["pandas.Series.reset_index"]),
        op("pandas.Series.reset_index", "pandas", pandas_version, "pandas.core.series", "Series.reset_index", "(name=None, **kwargs) -> DataFrame", "Move a Series index into columns and return a DataFrame.", ["series/pandas"], ["table/pandas-dataframe"], "adapt", ["series to dataframe", "flatten result"], ["pandas.DataFrame.to_json", "pandas.DataFrame.to_csv"]),
        op("pandas.DataFrame.to_json", "pandas", pandas_version, "pandas.core.generic", "DataFrame.to_json", "(path_or_buf=None, **kwargs) -> str | None", "Write a DataFrame as JSON records.", ["table/pandas-dataframe"], ["file/json-records"], "write", ["export json records", "save json"]),
        op("pandas.DataFrame.to_csv", "pandas", pandas_version, "pandas.core.generic", "DataFrame.to_csv", "(path_or_buf=None, **kwargs) -> str | None", "Write a DataFrame as comma-separated data.", ["table/pandas-dataframe"], ["file/csv"], "write", ["export csv", "save csv"]),
        op("pathlib.Path.read_text", "python-stdlib", stdlib_version, "pathlib", "Path.read_text", "(encoding=None, errors=None) -> str", "Read a text file into a string.", ["file/text"], ["text/plain"], "load", ["read text file"], ["json.loads", "re.findall"]),
        op("pathlib.Path.write_text", "python-stdlib", stdlib_version, "pathlib", "Path.write_text", "(data, encoding=None, errors=None) -> int", "Write a string to a text file.", ["text/plain", "text/json"], ["file/text"], "write", ["write text file"]),
        op("json.loads", "python-stdlib", stdlib_version, "json", "loads", "(s, **kwargs) -> object", "Deserialize JSON text to Python objects.", ["text/json", "text/plain"], ["object/python"], "parse", ["parse json", "json text object"], ["json.dumps"]),
        op("json.dumps", "python-stdlib", stdlib_version, "json", "dumps", "(obj, **kwargs) -> str", "Serialize a Python object to JSON text.", ["object/python"], ["text/json"], "serialize", ["serialize json", "object json text"], ["pathlib.Path.write_text"]),
        op("re.findall", "python-stdlib", stdlib_version, "re", "findall", "(pattern, string, flags=0) -> list", "Return all non-overlapping regular-expression matches.", ["text/plain"], ["list/string"], "extract", ["regex matches", "extract text pattern"]),
    ]

GRAPH_EDGE_FEATURES: dict[tuple[str, str], list[dict[str, Any]]] = defaultdict(list)
UPSTREAM_EMBEDDINGS: dict[str, dict[str, Any]] = {}
OPERATION_LEDGER_FEATURE_COUNTS: dict[str, dict[str, int]] = {}
LEDGER_CONTEXT: dict[str, Any] = {}

def normalized_port_type(annotation: str) -> str:
    tokens = [item for item in re.findall(r"[a-z0-9]+", str(annotation).casefold()) if item not in {"typing", "optional", "union", "none"}]
    return "annotation/python:" + "-".join(tokens) if tokens else "unknown/*"

def discover_feature_ledger_run() -> Path | None:
    candidates: list[Path] = []
    if FEATURE_LEDGER_HINT:
        hinted = Path(FEATURE_LEDGER_HINT)
        candidates.append(hinted if hinted.is_dir() else hinted.parent)
    roots = [Path("/kaggle/working/search_feature_experiments"), Path("./kaggle_working/search_feature_experiments")]
    if PROJECT_ROOT is not None:
        roots.extend([PROJECT_ROOT / "search_feature_experiments", PROJECT_ROOT.parent / "kaggle_working/search_feature_experiments"])
    for root in roots:
        if root.is_dir():
            candidates.extend(path for path in root.iterdir() if path.is_dir())
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        candidates.extend(path.parent for path in list(kaggle_input.rglob("feature_registry_snapshot.parquet"))[:50])
    valid = [path for path in dict.fromkeys(candidates) if (path / "operations.parquet").is_file() and (path / "feature_registry_snapshot.parquet").is_file()]
    return max(valid, key=lambda path: (path / "feature_registry_snapshot.parquet").stat().st_mtime) if valid else None

def parse_ledger_value(row: Mapping[str, Any]) -> Any:
    raw_json = row.get("value_json", "")
    if raw_json is not None and not (isinstance(raw_json, float) and np.isnan(raw_json)) and str(raw_json).strip():
        try:
            return json.loads(str(raw_json))
        except json.JSONDecodeError:
            pass
    return str(row.get("value_text", ""))

def load_feature_ledger_run(run_dir: Path) -> list[Operation]:
    operations_frame = pd.read_parquet(run_dir / "operations.parquet")
    feature_frame = pd.read_parquet(run_dir / "feature_registry_snapshot.parquet")
    entity_column = "entity_kind" if "entity_kind" in feature_frame else "entity_type"
    kind_column = "feature_kind" if "feature_kind" in feature_frame else "feature_family"
    required_operation_columns = {"operation_id", "package_name", "package_version", "module", "qualified_name", "kind", "signature", "docstring", "evidence_level"}
    missing = sorted(required_operation_columns - set(operations_frame.columns))
    if missing:
        raise ValueError(f"feature-ledger operations missing columns: {missing}")
    operation_id_set = set(operations_frame["operation_id"].astype(str))
    operation_features = feature_frame[(feature_frame[entity_column].astype(str) == "operation") & feature_frame["entity_id"].astype(str).isin(operation_id_set)].copy()
    feature_rows_by_operation: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for row in operation_features.to_dict(orient="records"):
        feature_rows_by_operation[str(row["entity_id"])].append(row)

    graph_next: dict[str, set[str]] = defaultdict(set)
    edges_path = run_dir / "candidate_edges.parquet"
    if edges_path.is_file():
        for edge in pd.read_parquet(edges_path).to_dict(orient="records"):
            source = str(edge["source_operation_id"]); target = str(edge["target_operation_id"])
            if source not in operation_id_set or target not in operation_id_set:
                continue
            record = {"relation": str(edge.get("relation", "unknown")), "directed": bool(edge.get("directed", True)), "verified": bool(edge.get("verified", False)), "generator": str(edge.get("generator", "feature_ledger")), "generator_revision": str(edge.get("generator_revision", "unknown"))}
            graph_next[source].add(target); GRAPH_EDGE_FEATURES[(source, target)].append(record)
            if not record["directed"]:
                graph_next[target].add(source); GRAPH_EDGE_FEATURES[(target, source)].append(record)

    loaded: list[Operation] = []
    for operation_row in operations_frame.sort_values("operation_id").to_dict(orient="records"):
        operation_id = str(operation_row["operation_id"])
        rows = feature_rows_by_operation.get(operation_id, [])
        labels: list[str] = []
        blocks: list[tuple[str, str]] = []
        input_types: list[str] = []
        output_types: list[str] = []
        description_parts = [str(operation_row.get("docstring", ""))]
        counts: dict[str, int] = defaultdict(int)
        for feature in rows:
            feature_kind = str(feature[kind_column]); namespace = str(feature.get("namespace", "")); value = parse_ledger_value(feature)
            counts[feature_kind] += 1
            if feature_kind in {"normalized_label", "label", "tag"} and isinstance(value, str):
                labels.append(value)
            elif feature_kind in {"blocking_key", "catalog_signal"} and isinstance(value, str):
                blocks.append((namespace, value))
            elif feature_kind == "input_port" and isinstance(value, dict):
                input_types.append(normalized_port_type(str(value.get("annotation", ""))))
            elif feature_kind == "output_port" and isinstance(value, dict):
                output_types.append(normalized_port_type(str(value.get("annotation", ""))))
            elif feature_kind in {"search_phrase", "semantic_phrase", "onion_layer", "catalog_representation"} and isinstance(value, str) and value:
                description_parts.append(value)
        stage_values = [value for namespace, value in blocks if namespace == "workflow_stage"]
        module = str(operation_row["module"])
        OPERATION_LEDGER_FEATURE_COUNTS[operation_id] = dict(counts)
        loaded.append(Operation(
            operation_id=operation_id, package_name=str(operation_row["package_name"]), package_version=str(operation_row["package_version"]),
            subpackage=module.rsplit(".", 1)[0] if "." in module else module, module=module, symbol=str(operation_row["qualified_name"]),
            kind=str(operation_row["kind"]), signature=str(operation_row["signature"]), description="\n".join(dict.fromkeys(part for part in description_parts if part))[:12000],
            input_types=tuple(dict.fromkeys(input_types)) or ("unknown/*",), output_types=tuple(dict.fromkeys(output_types)) or ("unknown/*",),
            stage=stage_values[0] if stage_values else "unknown", labels=tuple(dict.fromkeys(labels)), blocks=tuple(dict.fromkeys(blocks)),
            graph_next=tuple(sorted(graph_next.get(operation_id, set()))), evidence_level=str(operation_row["evidence_level"]), source=f"feature_ledger:{run_dir}",
        ))

    embedding_manifest_path = run_dir / "embedding_manifest.json"
    if embedding_manifest_path.is_file():
        for item in json.loads(embedding_manifest_path.read_text(encoding="utf-8")):
            plane_id = str(item["plane_id"]); key = f"ledger::{plane_id}"
            artifact = Path(str(item["artifact"]))
            if not artifact.is_file():
                artifact = run_dir / artifact.name
            UPSTREAM_EMBEDDINGS[key] = {**item, "artifact": str(artifact), "operation_ids": operations_frame["operation_id"].astype(str).tolist(), "run_dir": str(run_dir)}
    LEDGER_CONTEXT.update({"run_dir": str(run_dir), "operation_count": len(loaded), "feature_count": len(operation_features), "edge_count": sum(len(value) for value in GRAPH_EDGE_FEATURES.values()), "embedding_planes": sorted(UPSTREAM_EMBEDDINGS)})
    return loaded

def sqlite_candidates() -> list[Path]:
    paths: list[Path] = []
    if os.getenv("CATALOG_SQLITE"):
        paths.append(Path(os.environ["CATALOG_SQLITE"]))
    if PROJECT_ROOT is not None:
        paths.extend([PROJECT_ROOT / "catalog.sqlite3", PROJECT_ROOT / "data/catalog.sqlite3", PROJECT_ROOT / "reports/catalog.sqlite3"])
    for root in (Path("/kaggle/working"), Path("/kaggle/input")):
        if root.exists():
            paths.extend(list(root.glob("*catalog*.sqlite*"))[:20])
            paths.extend(list(root.glob("*/*catalog*.sqlite*"))[:20])
    return [path for path in dict.fromkeys(paths) if path.is_file()]

def load_sqlite_catalog(path: Path) -> list[Operation]:
    with sqlite3.connect(path) as connection:
        connection.row_factory = sqlite3.Row
        tables = {row[0] for row in connection.execute("SELECT name FROM sqlite_master WHERE type='table'")}
        if "operations" not in tables:
            return []
        package_python: dict[str, str] = {}
        if "packages" in tables:
            for row in connection.execute("SELECT package_id, requires_python FROM packages"):
                package_python[row["package_id"]] = row["requires_python"] or ">=3.9"
        signal_map: dict[str, list[tuple[str, str, str]]] = defaultdict(list)
        if "signals" in tables:
            for row in connection.execute("SELECT operation_id, signal_kind, namespace, value FROM signals"):
                signal_map[row["operation_id"]].append((row["signal_kind"], row["namespace"], row["value"]))
        loaded: list[Operation] = []
        for row in connection.execute("SELECT * FROM operations ORDER BY operation_id"):
            signals = signal_map.get(row["operation_id"], [])
            labels = tuple(value for kind, namespace, value in signals if kind == "label")
            blocks = tuple((namespace, value) for kind, namespace, value in signals if kind == "blocking_key")
            inputs = tuple(value for namespace, value in blocks if namespace == "input_artifact") or ("unknown/*",)
            outputs = tuple(value for namespace, value in blocks if namespace == "output_artifact") or ("unknown/*",)
            stages = [value for namespace, value in blocks if namespace == "workflow_stage"]
            module = row["module"]
            loaded.append(Operation(
                operation_id=row["operation_id"], package_name=row["package_name"],
                package_version=row["package_version"], subpackage=module.rsplit(".", 1)[0] if "." in module else module,
                module=module, symbol=row["qualified_name"], kind=row["kind"], signature=row["signature"],
                description=row["docstring"], input_types=inputs, output_types=outputs,
                stage=stages[0] if stages else "unknown", labels=labels, blocks=blocks,
                python_spec=package_python.get(row["package_id"], ">=3.9"),
                evidence_level=row["evidence_level"], source=f"sqlite:{path}",
            ))
        return loaded

def load_repository_projection() -> list[Operation]:
    if PROJECT_ROOT is None:
        return []
    try:
        from existing_code_reuse.capabilities import seed_capabilities
    except Exception as error:
        print("repository projection unavailable:", repr(error))
        return []
    loaded: list[Operation] = []
    for cap in seed_capabilities():
        module, _, symbol = cap.qualified_name.rpartition(".")
        blocks = [("workflow_stage", cap.workflow_stage)]
        blocks.extend(("input_artifact", item.artifact_type) for item in cap.inputs)
        blocks.extend(("output_artifact", item.artifact_type) for item in cap.outputs)
        loaded.append(Operation(
            operation_id=cap.capability_id, package_name=cap.package_name, package_version=cap.package_version,
            subpackage=module.rsplit(".", 1)[0] if "." in module else module, module=module or cap.package_name,
            symbol=symbol or cap.qualified_name, kind="function",
            signature="(" + ", ".join(f"{item.name}: {item.artifact_type}" for item in cap.inputs) + ") -> " + ", ".join(item.artifact_type for item in cap.outputs),
            description=" ".join([cap.purpose, *cap.limitations]), input_types=tuple(item.artifact_type for item in cap.inputs),
            output_types=tuple(item.artifact_type for item in cap.outputs), stage=cap.workflow_stage,
            labels=tuple(cap.aliases), blocks=tuple(blocks), evidence_level=cap.evidence_level, source="repository_verified_projection",
        ))
    return loaded

CATALOG_ORIGIN = "fallback_fixture"
CATALOG: list[Operation] = []
feature_run = discover_feature_ledger_run()
if feature_run is not None:
    try:
        CATALOG = load_feature_ledger_run(feature_run)
    except Exception as error:
        print(f"feature-ledger load failed for {feature_run}: {error!r}")
    if CATALOG:
        CATALOG_ORIGIN = f"feature_ledger:{feature_run}"
for sqlite_path in (() if CATALOG else sqlite_candidates()):
    try:
        CATALOG = load_sqlite_catalog(sqlite_path)
    except Exception as error:
        print(f"catalog load failed for {sqlite_path}: {error!r}")
        continue
    if CATALOG:
        CATALOG_ORIGIN = f"sqlite:{sqlite_path}"
        break
if not CATALOG:
    CATALOG = load_repository_projection()
    if CATALOG:
        CATALOG_ORIGIN = "repository_verified_projection"
if not CATALOG:
    CATALOG = fallback_catalog()
FALLBACK_AUGMENTED = CATALOG_ORIGIN.startswith("feature_ledger:")
if FALLBACK_AUGMENTED:
    existing_ids = {item.operation_id for item in CATALOG}
    CATALOG.extend(item for item in fallback_catalog() if item.operation_id not in existing_ids)
    CATALOG_ORIGIN += "+fallback_task_fixture"
for operation in CATALOG:
    for target_id in operation.graph_next:
        if target_id in {item.operation_id for item in CATALOG} and not GRAPH_EDGE_FEATURES[(operation.operation_id, target_id)]:
            GRAPH_EDGE_FEATURES[(operation.operation_id, target_id)].append({"relation": "declared_workflow_flow", "directed": True, "verified": operation.source == "fallback_fixture", "generator": operation.source, "generator_revision": "v1"})

operation_ids = [item.operation_id for item in CATALOG]
if len(operation_ids) != len(set(operation_ids)):
    raise ValueError("catalog operation IDs must be unique")
OPERATION_BY_ID = {item.operation_id: item for item in CATALOG}
print(f"catalog_origin={CATALOG_ORIGIN!r}, operations={len(CATALOG)}, subpackages={len({item.subpackage for item in CATALOG})}, ledger={json.dumps(LEDGER_CONTEXT, default=str)}")
pd.DataFrame([asdict(item) for item in CATALOG])[["operation_id", "subpackage", "stage", "input_types", "output_types", "source"]].head(20)

## Deterministic environment, port, and route checks

Ports are primary-artifact contracts. The adapter table is explicit and intentionally tiny; an embedding similarity never creates a compatibility edge.

In [ ]:
ADAPTERS: dict[tuple[str, str], str] = {
    ("text/plain", "text/json"): "identity.text_json_claim",
}

def port_match(actual: str, expected: str, allow_adapter: bool = False) -> tuple[bool, str | None]:
    if actual == expected:
        return True, None
    if actual == "unknown/*" or expected == "unknown/*":
        return False, None
    adapter = ADAPTERS.get((actual, expected)) if allow_adapter else None
    return adapter is not None, adapter

def environment_check(operation: Operation, goal: GoalIR) -> tuple[bool, tuple[str, ...]]:
    reasons: list[str] = []
    try:
        if operation.python_spec and Version(goal.python_version) not in SpecifierSet(operation.python_spec):
            reasons.append(f"python {goal.python_version} not in {operation.python_spec}")
    except InvalidSpecifier:
        reasons.append(f"invalid package Python specifier: {operation.python_spec}")
    if "any" not in operation.platforms and goal.platform not in operation.platforms:
        reasons.append(f"platform {goal.platform} not in {operation.platforms}")
    if not goal.network_allowed and "network" in operation.effects:
        reasons.append("network effect prohibited by GoalIR")
    return not reasons, tuple(reasons)

def ordered_stage_progress(required: Sequence[str], covered: Sequence[str]) -> int:
    position = 0
    for stage in covered:
        if position < len(required) and stage == required[position]:
            position += 1
    return position

def validate_operation_ids(operation_ids: Sequence[str], catalog: Mapping[str, Operation]) -> tuple[bool, tuple[str, ...]]:
    unknown = tuple(item for item in operation_ids if item not in catalog)
    return not unknown, unknown

def validate_route(goal: GoalIR, operation_ids: Sequence[str], allow_adapters: bool = False) -> ValidationResult:
    ids_valid, unknown = validate_operation_ids(operation_ids, OPERATION_BY_ID)
    if not ids_valid:
        return ValidationResult(False, "unknown_operation_id", tuple(f"unknown operation: {item}" for item in unknown))
    artifact = goal.initial_artifact
    stages: list[str] = []
    reasons: list[str] = []
    for operation_id in operation_ids:
        operation = OPERATION_BY_ID[operation_id]
        env_ok, env_reasons = environment_check(operation, goal)
        if not env_ok:
            reasons.extend(f"{operation_id}: {item}" for item in env_reasons)
            break
        matches = [port_match(artifact, expected, allow_adapters) for expected in operation.input_types]
        if not any(item[0] for item in matches):
            reasons.append(f"{operation_id}: no input port accepts {artifact}; expected {operation.input_types}")
            break
        artifact = operation.output_types[0]
        stages.append(operation.stage)
    if reasons:
        return ValidationResult(False, "incompatible", tuple(reasons), artifact, tuple(stages))
    stage_progress = ordered_stage_progress(goal.required_stages, stages)
    if stage_progress != len(goal.required_stages):
        reasons.append(f"covered required stages {stage_progress}/{len(goal.required_stages)} in order")
    if artifact != goal.required_output:
        reasons.append(f"final artifact {artifact!r} != required {goal.required_output!r}")
    return ValidationResult(not reasons, "compatible" if not reasons else "incomplete", tuple(reasons), artifact, tuple(stages))

def validate_recipe(goal: GoalIR, recipe: RecipeIR) -> ValidationResult:
    if recipe.decision != "reuse":
        return ValidationResult(True, recipe.decision, ())
    return validate_route(goal, recipe.operation_ids)

## Hybrid subpackage/symbol retrieval

Each score component is retained as candidate provenance. The hierarchical term aggregates symbol evidence within `(package, subpackage)` and adds a bounded subpackage bonus. Dense encoders are loaded lazily and document vectors are cached by exact model/field configuration.

In [ ]:
@dataclass(frozen=True)
class RetrievalArm:
    arm_id: str
    lexical: bool = True
    labels: bool = False
    blocks: bool = False
    dense: bool = False
    graph: bool = False
    compatibility: bool = False
    descriptions: bool = True
    dense_model_id: str | None = None
    dense_fields: tuple[str, ...] = ("description",)
    top_k: int = TOP_K
    abstain_threshold: float = ABSTAIN_THRESHOLD

def normalize_scores(values: np.ndarray) -> np.ndarray:
    if values.size == 0 or float(values.max(initial=0.0)) <= 0.0:
        return np.zeros_like(values, dtype=float)
    return values / float(values.max())

def operation_text(operation: Operation, include_description: bool, fields: Sequence[str] | None = None) -> str:
    values = {
        "identity": " ".join([operation.operation_id, operation.package_name, operation.subpackage, operation.module, operation.symbol]),
        "signature": operation.signature,
        "description": operation.description,
        "labels": " ".join(operation.labels),
        "blocks": " ".join(f"{key} {value}" for key, value in operation.blocks),
    }
    selected = tuple(fields) if fields is not None else (("identity", "signature", "description") if include_description else ("identity", "signature"))
    return "\n".join(values[item] for item in selected if item in values and values[item])

class DenseRegistry:
    def __init__(self) -> None:
        self.models: dict[str, Any] = {}
        self.document_vectors: dict[tuple[str, tuple[str, ...], str], np.ndarray] = {}
        self.ledger_matrices: dict[str, np.ndarray] = {}

    def scores(self, model_id: str, fields: tuple[str, ...], query: str, catalog: Sequence[Operation]) -> np.ndarray:
        if model_id.startswith("ledger::"):
            return self._ledger_scores(model_id, query, catalog)
        from sentence_transformers import SentenceTransformer
        if model_id not in self.models:
            self.models[model_id] = SentenceTransformer(model_id, device=DENSE_DEVICE, trust_remote_code=False)
        model = self.models[model_id]
        catalog_digest = stable_digest([(item.operation_id, operation_text(item, True, fields)) for item in catalog])
        key = (model_id, fields, catalog_digest)
        if key not in self.document_vectors:
            docs = [operation_text(item, True, fields) for item in catalog]
            self.document_vectors[key] = np.asarray(model.encode(docs, normalize_embeddings=True, batch_size=DENSE_BATCH_SIZE, show_progress_bar=False))
        query_vector = np.asarray(model.encode([query], normalize_embeddings=True, show_progress_bar=False))
        return np.asarray(self.document_vectors[key] @ query_vector[0], dtype=float)

    def _ledger_scores(self, ledger_key: str, query: str, catalog: Sequence[Operation]) -> np.ndarray:
        if ledger_key not in UPSTREAM_EMBEDDINGS:
            raise KeyError(f"unknown upstream embedding plane: {ledger_key}")
        item = UPSTREAM_EMBEDDINGS[ledger_key]
        artifact = Path(item["artifact"])
        if not artifact.is_file():
            raise FileNotFoundError(f"upstream embedding artifact is unavailable: {artifact}")
        if ledger_key not in self.ledger_matrices:
            self.ledger_matrices[ledger_key] = np.load(artifact, allow_pickle=False)
        matrix = self.ledger_matrices[ledger_key]
        spec = dict(item.get("model_spec", {})); backend = spec.get("backend")
        if backend == "sklearn_hashing":
            vectorizer = __import__("sklearn.feature_extraction.text", fromlist=["HashingVectorizer"]).HashingVectorizer(
                n_features=int(spec["dimension"]), analyzer=str(spec.get("analyzer", "word")),
                ngram_range=tuple(spec.get("ngram_range", (1, 2))), alternate_sign=bool(spec.get("alternate_sign", False)), norm="l2",
            )
            query_vector = vectorizer.transform([query]).toarray()[0]
        elif backend == "sentence_transformers":
            from sentence_transformers import SentenceTransformer
            cache_key = str(spec["model_id"]) + "@" + str(spec.get("revision", ""))
            if cache_key not in self.models:
                self.models[cache_key] = SentenceTransformer(str(spec["model_id"]), revision=spec.get("revision") or None, device=DENSE_DEVICE, trust_remote_code=False)
            query_vector = np.asarray(self.models[cache_key].encode([query], normalize_embeddings=True, show_progress_bar=False))[0]
        else:
            raise ValueError(f"unsupported upstream embedding backend: {backend!r}")
        if matrix.shape[0] != len(item["operation_ids"]):
            raise ValueError(f"upstream matrix row mismatch for {ledger_key}")
        upstream_scores = np.asarray(matrix @ query_vector, dtype=float)
        score_by_id = dict(zip(item["operation_ids"], upstream_scores, strict=True))
        return np.asarray([score_by_id.get(operation.operation_id, 0.0) for operation in catalog], dtype=float)

DENSE_REGISTRY = DenseRegistry()

def inferred_goal_blocks(goal: GoalIR) -> set[tuple[str, str]]:
    result = set(goal.query_blocks)
    result.add(("input_artifact", goal.initial_artifact))
    result.add(("output_artifact", goal.required_output))
    result.update(("workflow_stage", item) for item in goal.required_stages)
    return result

def retrieve(goal: GoalIR, arm: RetrievalArm) -> tuple[list[Candidate], float]:
    started = time.perf_counter()
    n = len(CATALOG)
    components: dict[str, np.ndarray] = {}
    details: list[dict[str, Any]] = [defaultdict(list) for _ in CATALOG]

    if arm.lexical:
        documents = [operation_text(item, arm.descriptions) for item in CATALOG]
        word = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, strip_accents="unicode")
        char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True)
        word_matrix = word.fit_transform(documents)
        char_matrix = char.fit_transform(documents)
        components["lexical"] = normalize_scores(0.6 * (word_matrix @ word.transform([goal.prompt]).T).toarray().ravel() + 0.4 * (char_matrix @ char.transform([goal.prompt]).T).toarray().ravel())

    if arm.labels:
        query_terms = {item.casefold() for item in goal.query_labels}
        query_terms.update(re.findall(r"[a-z0-9_-]+", goal.prompt.casefold()))
        raw = []
        for index, operation in enumerate(CATALOG):
            op_terms = set(re.findall(r"[a-z0-9_-]+", " ".join(operation.labels).casefold()))
            overlap = sorted(query_terms & op_terms)
            details[index]["matched_labels"] = overlap
            raw.append(len(overlap) / max(1, len(query_terms | op_terms)))
        components["labels"] = normalize_scores(np.asarray(raw, dtype=float))

    if arm.blocks:
        requested = inferred_goal_blocks(goal)
        raw = []
        for index, operation in enumerate(CATALOG):
            matched = sorted(requested & set(operation.blocks))
            details[index]["matched_blocks"] = matched
            raw.append(len(matched) / max(1, len(requested)))
        components["blocks"] = normalize_scores(np.asarray(raw, dtype=float))

    if arm.graph:
        required = set(goal.required_stages)
        raw = np.zeros(n, dtype=float)
        by_id = {item.operation_id: index for index, item in enumerate(CATALOG)}
        seed_signal = np.maximum.reduce(list(components.values())) if components else np.zeros(n, dtype=float)
        for index, operation in enumerate(CATALOG):
            if operation.stage in required:
                raw[index] += 0.7
            for target_id in operation.graph_next:
                target = OPERATION_BY_ID.get(target_id)
                edge_features = GRAPH_EDGE_FEATURES.get((operation.operation_id, target_id), [])
                details[index]["graph_edges"].append({"target_operation_id": target_id, "features": edge_features})
                raw[index] += min(0.2, 0.04 * max(1, len(edge_features)))
                if target_id in by_id:
                    raw[by_id[target_id]] += 0.35 * seed_signal[index]
                if target is not None and target.stage in required:
                    raw[index] += 0.3
                    if target_id in by_id:
                        raw[by_id[target_id]] += 0.2
        components["graph"] = normalize_scores(raw)

    if arm.dense:
        if not arm.dense_model_id:
            raise ValueError(f"{arm.arm_id}: dense arm requires dense_model_id")
        components["dense"] = normalize_scores(DENSE_REGISTRY.scores(arm.dense_model_id, arm.dense_fields, goal.prompt, CATALOG))

    active = tuple(components)
    combined = sum(components.values(), start=np.zeros(n, dtype=float)) / max(1, len(active))
    allowed = np.ones(n, dtype=bool)
    if arm.compatibility:
        for index, operation in enumerate(CATALOG):
            ok, reasons = environment_check(operation, goal)
            allowed[index] = ok
            details[index]["compatibility"] = {"allowed": ok, "reasons": reasons}

    group_max: dict[tuple[str, str], float] = defaultdict(float)
    for index, operation in enumerate(CATALOG):
        if allowed[index]:
            group_max[(operation.package_name, operation.subpackage)] = max(group_max[(operation.package_name, operation.subpackage)], float(combined[index]))
    hierarchical = combined.copy()
    for index, operation in enumerate(CATALOG):
        if allowed[index]:
            bonus = 0.10 * group_max[(operation.package_name, operation.subpackage)]
            hierarchical[index] += bonus
            details[index]["subpackage_bonus"] = bonus
        else:
            hierarchical[index] = -1.0

    order = sorted(range(n), key=lambda index: (-float(hierarchical[index]), CATALOG[index].operation_id))
    candidates: list[Candidate] = []
    for index in order:
        if hierarchical[index] <= 0 or len(candidates) >= arm.top_k:
            break
        operation = CATALOG[index]
        provenance = {
            "catalog_origin": CATALOG_ORIGIN,
            "package": operation.package_name,
            "subpackage": operation.subpackage,
            "module": operation.module,
            "symbol": operation.symbol,
            "operation_id": operation.operation_id,
            "ledger_feature_counts": OPERATION_LEDGER_FEATURE_COUNTS.get(operation.operation_id, {}),
            "components": {name: float(values[index]) for name, values in components.items()},
            **details[index],
        }
        candidates.append(Candidate(operation.operation_id, float(hierarchical[index]), len(candidates) + 1, operation.subpackage, provenance))
    return candidates, (time.perf_counter() - started) * 1000.0

## Multi-step deterministic composition

The planner searches only retrieved operation IDs, respects ordered workflow stages and exact ports, and prefers higher measured candidate scores. It emits `no_reuse` only after a sufficiently strong retrieval has no valid route; weak retrieval emits `abstain`.

In [ ]:
def advance_stage(required: Sequence[str], position: int, operation_stage: str) -> int:
    return position + 1 if position < len(required) and required[position] == operation_stage else position

def compose_route(goal: GoalIR, candidates: Sequence[Candidate], arm: RetrievalArm) -> RecipeIR:
    if not candidates or candidates[0].score < arm.abstain_threshold:
        return RecipeIR("abstain", confidence=candidates[0].score if candidates else 0.0, rationale="top retrieval score below threshold")
    scores = {item.operation_id: item.score for item in candidates}
    candidate_operations = [OPERATION_BY_ID[item.operation_id] for item in candidates]
    queue = deque([(goal.initial_artifact, 0, tuple(), 0.0)])
    best_seen: dict[tuple[str, int, int], float] = {}
    valid_routes: list[tuple[float, tuple[str, ...]]] = []
    while queue:
        artifact, stage_position, route, route_score = queue.popleft()
        if artifact == goal.required_output and stage_position == len(goal.required_stages):
            valid_routes.append((route_score / max(1, len(route)), route))
            continue
        if len(route) >= goal.max_steps:
            continue
        for operation in candidate_operations:
            if operation.operation_id in route:
                continue
            env_ok, _ = environment_check(operation, goal)
            if not env_ok or not any(port_match(artifact, expected)[0] for expected in operation.input_types):
                continue
            next_stage = advance_stage(goal.required_stages, stage_position, operation.stage)
            if operation.stage in goal.required_stages and next_stage == stage_position:
                continue  # do not reorder required workflow stages
            next_artifact = operation.output_types[0]
            next_route = route + (operation.operation_id,)
            next_score = route_score + scores[operation.operation_id]
            state = (next_artifact, next_stage, len(next_route))
            if best_seen.get(state, -1.0) >= next_score:
                continue
            best_seen[state] = next_score
            queue.append((next_artifact, next_stage, next_route, next_score))
    if not valid_routes:
        return RecipeIR("no_reuse", confidence=min(1.0, candidates[0].score), rationale="no compatible route in retrieved supply")
    valid_routes.sort(key=lambda item: (-item[0], len(item[1]), item[1]))
    score, route = valid_routes[0]
    recipe = RecipeIR("reuse", route, min(1.0, score), rationale="highest-scoring compatible route")
    validation = validate_recipe(goal, recipe)
    if not validation.valid:
        raise AssertionError(f"planner emitted an invalid route: {validation}")
    return recipe

## Optional model-agnostic Hugging Face JSON proposer

For the default Gemma 4 checkpoint, the adapter follows Google's [official inference pattern](https://ai.google.dev/gemma/docs/capabilities/thinking): `AutoProcessor` plus `AutoModelForMultimodalLM`, `dtype="auto"`, and `device_map="auto"`. `HF_MODEL_ID` remains configurable within compatible checkpoints. Thinking is disabled for this bounded JSON-selection task to limit output cost. JSON is constrained by a strict post-generation schema and operation allowlist; this is not a claim of token-level grammar decoding. Invalid output is recorded as an error/abstention, never repaired into a successful result.

In [ ]:
def extract_first_json_object(text: str) -> Mapping[str, Any]:
    decoder = json.JSONDecoder()
    for index, character in enumerate(text):
        if character != "{":
            continue
        try:
            value, _ = decoder.raw_decode(text[index:])
        except json.JSONDecodeError:
            continue
        if isinstance(value, dict):
            return value
    raise ValueError("model output did not contain a JSON object")

class HuggingFaceJSONProposer:
    def __init__(self, model_id: str) -> None:
        from transformers import AutoModelForMultimodalLM, AutoProcessor
        self.model_id = model_id
        token = os.getenv("HF_TOKEN") or None
        self.model = AutoModelForMultimodalLM.from_pretrained(model_id, dtype="auto", device_map="auto", token=token)
        self.processor = AutoProcessor.from_pretrained(model_id, token=token)

    def _generate(self, instruction: str, payload: Mapping[str, Any]) -> Mapping[str, Any]:
        messages = [
            {"role": "system", "content": instruction + " Return one JSON object and no markdown."},
            {"role": "user", "content": json.dumps(payload, default=str)},
        ]
        prompt = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        encoded = self.processor(text=prompt, return_tensors="pt").to(self.model.device)
        generated = self.model.generate(**encoded, max_new_tokens=HF_MAX_NEW_TOKENS, do_sample=False)
        new_tokens = generated[0, encoded["input_ids"].shape[1]:]
        return extract_first_json_object(self.processor.decode(new_tokens, skip_special_tokens=True))

    def infer_goal(self, prompt: str, hints: Mapping[str, Any]) -> GoalIR:
        instruction = ("Fill GoalIR using only keys prompt, initial_artifact, required_output, required_stages, "
                       "query_labels, query_blocks, python_version, platform, network_allowed, max_steps. Preserve supplied hints.")
        return GoalIR.from_mapping(self._generate(instruction, {"prompt": prompt, "hints": hints}))

    def propose_recipe(self, goal: GoalIR, candidates: Sequence[Candidate]) -> RecipeIR:
        allowed_ids = {item.operation_id for item in candidates}
        payload = {
            "goal": asdict(goal),
            "candidates": [
                {"operation_id": item.operation_id, "score": item.score, "operation": asdict(OPERATION_BY_ID[item.operation_id])}
                for item in candidates
            ],
            "schema": {"decision": "reuse|no_reuse|abstain", "operation_ids": ["allowed IDs only"], "confidence": "0..1", "rationale": "short"},
        }
        raw = self._generate("Select a compatible ordered route. Never invent an operation ID.", payload)
        recipe = RecipeIR.from_mapping(raw, proposer=f"hf:{self.model_id}")
        unknown = set(recipe.operation_ids) - allowed_ids
        if unknown:
            raise ValueError(f"model selected IDs outside candidate allowlist: {sorted(unknown)}")
        validation = validate_recipe(goal, recipe)
        if not validation.valid:
            raise ValueError(f"model recipe failed deterministic validation: {validation}")
        return recipe

HF_PROPOSER = None
HF_LOAD_ERROR = None
if USE_HF_ROUTER:
    try:
        HF_PROPOSER = HuggingFaceJSONProposer(HF_MODEL_ID)
    except Exception as error:
        HF_LOAD_ERROR = repr(error)
        print("HF proposer unavailable; the error will be recorded:", HF_LOAD_ERROR)

## Frozen lab tasks

These tasks are deliberately small. They test direct reuse, a six-symbol composition, a two-step standard-library route, a hard negative, and a true no-route case. Replace or extend them with a versioned benchmark file for substantive claims.

In [ ]:
@dataclass(frozen=True)
class RoutingTask:
    task_id: str
    goal: GoalIR
    acceptable_routes: tuple[tuple[str, ...], ...] = ()
    expect_no_reuse: bool = False
    hard_negatives: tuple[str, ...] = ()

def fallback_tasks() -> list[RoutingTask]:
    return [
        RoutingTask("csv_load", GoalIR("Load a comma-separated transaction file into a dataframe", "file/csv", "table/pandas-dataframe", ("load",), ("load csv",), (("input_artifact", "file/csv"),)), (("pandas.read_csv",),), hard_negatives=("pandas.read_json",)),
        RoutingTask("grouped_json", GoalIR("Read CSV, fill missing values, group by category, sum amounts, flatten, and write JSON records", "file/csv", "file/json-records", ("load", "missing-values", "group", "aggregate", "adapt", "write"), ("csv", "grouped sum", "json records"), max_steps=8), (("pandas.read_csv", "pandas.DataFrame.fillna", "pandas.DataFrame.groupby", "pandas.core.groupby.DataFrameGroupBy.sum", "pandas.Series.reset_index", "pandas.DataFrame.to_json"),)),
        RoutingTask("json_text_file", GoalIR("Read a text file containing JSON and deserialize it to a Python object", "file/text", "object/python", ("load", "parse"), ("read text", "parse json"), max_steps=3), (("pathlib.Path.read_text", "json.loads"),)),
        RoutingTask("json_round_trip_file", GoalIR("Serialize a Python object as JSON text and write it to a text file", "object/python", "file/text", ("serialize", "write"), ("serialize json", "write text"), max_steps=3), (("json.dumps", "pathlib.Path.write_text"),)),
        RoutingTask("csv_reader_hard_negative", GoalIR("Read CSV input; reject JSON readers", "file/csv", "table/pandas-dataframe", ("load",), ("csv reader",)), (("pandas.read_csv",),), hard_negatives=("pandas.read_json",)),
        RoutingTask("private_ledger_v7", GoalIR("Apply undocumented private company ledger semantics version 7", "file/csv", "file/company-ledger-v7", ("company-policy-v7",), ("private ledger",), (("output_artifact", "file/company-ledger-v7"),)), expect_no_reuse=True),
    ]

# Repository catalogs can be much larger, but the fallback task IDs are scored only when their gold operations exist.
TASKS = [task for task in fallback_tasks() if task.expect_no_reuse or any(all(op_id in OPERATION_BY_ID for op_id in route) for route in task.acceptable_routes)]
if not TASKS:
    raise RuntimeError("no benchmark tasks are compatible with the loaded catalog; supply a task adapter or use the fallback catalog")
print(f"tasks={len(TASKS)}, task_digest={stable_digest([asdict(item) for item in TASKS])}")

## Ablation grid and metrics

The light grid isolates lexical fields/descriptions, labels, deterministic blocks, graph signals, and compatibility, then combines them. Dense-only and full+dense arms are always represented in the configuration manifest; with `RUN_DENSE_ABLATIONS=0` they are honestly marked skipped. `medium`/`full` expands the Boolean mixture grid.

In [ ]:
def build_ablation_grid() -> list[RetrievalArm]:
    curated = [
        RetrievalArm("lexical_identity_signature", descriptions=False),
        RetrievalArm("lexical_with_descriptions", descriptions=True),
        RetrievalArm("labels_blocks", lexical=False, labels=True, blocks=True, descriptions=False),
        RetrievalArm("lexical_labels_blocks", labels=True, blocks=True),
        RetrievalArm("lexical_labels_blocks_graph", labels=True, blocks=True, graph=True),
        RetrievalArm("lexical_labels_blocks_compat", labels=True, blocks=True, compatibility=True),
        RetrievalArm("full_sparse_structural", labels=True, blocks=True, graph=True, compatibility=True),
    ]
    arms = list(curated)
    if ABLATION_SCALE in {"medium", "full"}:
        limit = 32 if ABLATION_SCALE == "medium" else 64
        for lexical, labels, blocks, graph, compatibility, descriptions in list(product([False, True], repeat=6))[:limit]:
            if not any((lexical, labels, blocks, graph)):
                continue
            arm_id = "mix_" + "_".join(f"{name}{int(value)}" for name, value in [("lex", lexical), ("lab", labels), ("blk", blocks), ("gra", graph), ("cmp", compatibility), ("desc", descriptions)])
            arms.append(RetrievalArm(arm_id, lexical=lexical, labels=labels, blocks=blocks, graph=graph, compatibility=compatibility, descriptions=descriptions))
    for model_id in DENSE_MODEL_IDS:
        model_slug = re.sub(r"[^a-z0-9]+", "-", model_id.casefold()).strip("-")[-48:]
        for fields in (("description",), ("identity", "signature", "description"), ("labels", "blocks", "description")):
            field_slug = "-".join(item[:4] for item in fields)
            arms.append(RetrievalArm(f"dense_{model_slug}_{field_slug}", lexical=False, dense=True, dense_model_id=model_id, dense_fields=fields))
            arms.append(RetrievalArm(f"full_dense_{model_slug}_{field_slug}", labels=True, blocks=True, dense=True, graph=True, compatibility=True, dense_model_id=model_id, dense_fields=fields))
    for ledger_key, item in sorted(UPSTREAM_EMBEDDINGS.items()):
        plane_slug = re.sub(r"[^a-z0-9]+", "-", ledger_key.casefold()).strip("-")
        fields = tuple(str(value) for value in item.get("fields", ("description",)))
        arms.append(RetrievalArm(f"upstream_{plane_slug}", lexical=False, dense=True, dense_model_id=ledger_key, dense_fields=fields))
        arms.append(RetrievalArm(f"full_upstream_{plane_slug}", labels=True, blocks=True, dense=True, graph=True, compatibility=True, dense_model_id=ledger_key, dense_fields=fields))
    deduped = {arm.arm_id: arm for arm in arms}
    return list(deduped.values())

def best_route_node_recall(selected: Sequence[str], acceptable: Sequence[Sequence[str]]) -> float:
    if not acceptable:
        return 0.0
    selected_set = set(selected)
    return max(len(selected_set & set(route)) / len(route) for route in acceptable)

def reciprocal_rank(candidate_ids: Sequence[str], relevant: set[str]) -> float:
    for rank, operation_id in enumerate(candidate_ids, start=1):
        if operation_id in relevant:
            return 1.0 / rank
    return 0.0

def evaluate_task(task: RoutingTask, arm: RetrievalArm) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    candidates, retrieval_ms = retrieve(task.goal, arm)
    plan_started = time.perf_counter()
    recipe = compose_route(task.goal, candidates, arm)
    planning_ms = (time.perf_counter() - plan_started) * 1000.0
    validation = validate_recipe(task.goal, recipe)
    candidate_ids = [item.operation_id for item in candidates]
    relevant = set().union(*(set(route) for route in task.acceptable_routes)) if task.acceptable_routes else set()
    exact_route = recipe.decision == "reuse" and any(tuple(recipe.operation_ids) == tuple(route) for route in task.acceptable_routes)
    expected_decision = "no_reuse" if task.expect_no_reuse else "reuse"
    hard_negative_violation = bool((set(candidate_ids[:5]) | set(recipe.operation_ids)) & set(task.hard_negatives))
    row = {
        "arm_id": arm.arm_id, "task_id": task.task_id, "status": "measured",
        "expected_decision": expected_decision, "decision": recipe.decision,
        "decision_correct": recipe.decision == expected_decision,
        "route_valid": validation.valid, "exact_route_match": exact_route,
        "route_node_recall": best_route_node_recall(recipe.operation_ids, task.acceptable_routes),
        "candidate_recall_at_k": best_route_node_recall(candidate_ids, task.acceptable_routes),
        "mrr": reciprocal_rank(candidate_ids, relevant),
        "hard_negative_violation": hard_negative_violation,
        "candidate_count": len(candidates), "top_score": candidates[0].score if candidates else 0.0,
        "retrieval_latency_ms": retrieval_ms, "planning_latency_ms": planning_ms,
        "total_latency_ms": retrieval_ms + planning_ms,
        "operation_ids": recipe.operation_ids, "recipe_confidence": recipe.confidence,
        "validation_status": validation.status, "validation_reasons": validation.reasons,
    }
    provenance = [
        {"arm_id": arm.arm_id, "task_id": task.task_id, **asdict(candidate)}
        for candidate in candidates
    ]
    return row, provenance

ARMS = build_ablation_grid()
print(f"ablation_arms={len(ARMS)}, dense_enabled={RUN_DENSE_ABLATIONS}")
pd.DataFrame([asdict(item) for item in ARMS]).head(20)

In [ ]:
task_rows: list[dict[str, Any]] = []
provenance_rows: list[dict[str, Any]] = []
arm_status_rows: list[dict[str, Any]] = []

for arm in ARMS:
    if arm.dense and not RUN_DENSE_ABLATIONS:
        arm_status_rows.append({"arm_id": arm.arm_id, "status": "skipped_dense_disabled", "error": None})
        continue
    print("running", arm.arm_id)
    arm_started = time.perf_counter()
    try:
        for task in TASKS:
            row, provenance = evaluate_task(task, arm)
            task_rows.append(row)
            provenance_rows.extend(provenance)
    except Exception as error:
        arm_status_rows.append({"arm_id": arm.arm_id, "status": "failed", "error": repr(error), "elapsed_ms": (time.perf_counter() - arm_started) * 1000.0})
        continue
    arm_status_rows.append({"arm_id": arm.arm_id, "status": "measured", "error": None, "elapsed_ms": (time.perf_counter() - arm_started) * 1000.0})

TASK_RESULTS = pd.DataFrame(task_rows)
ARM_STATUS = pd.DataFrame(arm_status_rows)
if TASK_RESULTS.empty:
    SUMMARY = ARM_STATUS.copy()
else:
    SUMMARY = TASK_RESULTS.groupby("arm_id", as_index=False).agg(
        measured_tasks=("task_id", "count"),
        decision_accuracy=("decision_correct", "mean"),
        valid_route_rate=("route_valid", "mean"),
        exact_route_rate=("exact_route_match", "mean"),
        mean_route_node_recall=("route_node_recall", "mean"),
        mean_candidate_recall_at_k=("candidate_recall_at_k", "mean"),
        mean_mrr=("mrr", "mean"),
        hard_negative_violation_rate=("hard_negative_violation", "mean"),
        mean_latency_ms=("total_latency_ms", "mean"),
        p95_latency_ms=("total_latency_ms", lambda values: float(np.percentile(values, 95))),
    ).merge(ARM_STATUS, on="arm_id", how="outer")
SUMMARY.sort_values(["status", "decision_accuracy", "mean_route_node_recall"], ascending=[True, False, False], na_position="last")

## Optional one-task small-model proposal

This arm is separate from the retrieval ablation table because loading/generating with a small model changes the resource budget. Its measured receipt includes errors. It may select only candidates returned by the strongest measured non-dense arm, and the deterministic route validator remains final.

In [ ]:
HF_RECEIPT: dict[str, Any] = {"enabled": USE_HF_ROUTER, "model_id": HF_MODEL_ID, "status": "not_requested"}
if USE_HF_ROUTER:
    if HF_PROPOSER is None:
        HF_RECEIPT.update({"status": "load_failed", "error": HF_LOAD_ERROR})
    else:
        sample_task = TASKS[0]
        model_arm = RetrievalArm("hf_candidate_supply", labels=True, blocks=True, graph=True, compatibility=True)
        model_candidates, retrieval_ms = retrieve(sample_task.goal, model_arm)
        started = time.perf_counter()
        try:
            model_recipe = HF_PROPOSER.propose_recipe(sample_task.goal, model_candidates)
            validation = validate_recipe(sample_task.goal, model_recipe)
            HF_RECEIPT.update({
                "status": "measured", "task_id": sample_task.task_id,
                "recipe": asdict(model_recipe), "validation": asdict(validation),
                "retrieval_latency_ms": retrieval_ms,
                "generation_latency_ms": (time.perf_counter() - started) * 1000.0,
            })
        except Exception as error:
            HF_RECEIPT.update({"status": "failed", "task_id": sample_task.task_id, "error": repr(error), "generation_latency_ms": (time.perf_counter() - started) * 1000.0})
print(json.dumps(HF_RECEIPT, indent=2, default=str))

## Persist measured receipts

The manifest hashes the exact catalog, tasks, and arm configurations used in this run. JSONL retains per-task decisions and full candidate provenance; CSV is only a convenient projection.

In [ ]:
def write_jsonl(path: Path, rows: Iterable[Mapping[str, Any]]) -> None:
    path.write_text("".join(json.dumps(row, sort_keys=True, default=str) + "\n" for row in rows), encoding="utf-8")

TASK_RESULTS.to_csv(OUTPUT_DIR / "ablation_task_results.csv", index=False)
SUMMARY.to_csv(OUTPUT_DIR / "ablation_summary.csv", index=False)
ARM_STATUS.to_csv(OUTPUT_DIR / "ablation_arm_status.csv", index=False)
write_jsonl(OUTPUT_DIR / "ablation_task_results.jsonl", task_rows)
write_jsonl(OUTPUT_DIR / "candidate_provenance.jsonl", provenance_rows)
(OUTPUT_DIR / "hf_router_receipt.json").write_text(json.dumps(HF_RECEIPT, indent=2, sort_keys=True, default=str), encoding="utf-8")
manifest = {
    "created_at_unix": time.time(),
    "lab_config": LAB_CONFIG,
    "catalog_origin": CATALOG_ORIGIN,
    "feature_ledger_context": LEDGER_CONTEXT,
    "catalog_count": len(CATALOG),
    "catalog_digest": stable_digest([asdict(item) for item in CATALOG]),
    "task_count": len(TASKS),
    "task_digest": stable_digest([asdict(item) for item in TASKS]),
    "arm_count": len(ARMS),
    "arm_digest": stable_digest([asdict(item) for item in ARMS]),
    "measured_row_count": len(task_rows),
    "candidate_provenance_row_count": len(provenance_rows),
    "model_inference_measured": HF_RECEIPT.get("status") == "measured",
    "model_inference_status": HF_RECEIPT.get("status"),
    "status_counts": ARM_STATUS["status"].value_counts().to_dict() if not ARM_STATUS.empty else {},
}
(OUTPUT_DIR / "lab_manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True, default=str), encoding="utf-8")
print("wrote:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {path.name}: {path.stat().st_size:,} bytes")

## Reading the results

Treat this notebook as an experiment harness, not a benchmark claim. Prefer configurations that improve compatibility-aware route recall and decision accuracy without increasing hard-negative violations, latency, or abstention errors. Inspect `candidate_provenance.jsonl` whenever aggregate scores move: it records whether the gain came from description leakage, a deterministic block, subpackage aggregation, graph expansion, dense similarity, or a compatibility gate. Scale to a frozen 100–250 package/task corpus before tuning model weights, and retain failures as first-class data.